# K3 — Derived outputs

This page checks phase K3: one summer4 `OutputSet` reproduces **every** derived output the
original model requested — including the `save_results=False` intermediates — by the same
name, from one shared compartment save and one save per flow.

**Claims to check.**

1. The port's output names are exactly the original's (674 without, 675 with age mixing).
2. Every output matches the original at every year to 1e-5 relative (floor 1e-8).
3. The calibration targets (TB incidence, notifications, TST/PEARL/CXR positivity,
   % subclinical, mixing distance) overlap the original line for line.

In [ ]:
import sys

import numpy as np
import pandas as pd
import plotly.io as pio
import yaml

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

from kiribati_tb.model import ModelConfig, compile_model
from kiribati_tb.outputs import build_outputs, run_outputs
from kiribati_tb.paths import GOLDEN, REPO_ROOT

sys.path.insert(0, str(REPO_ROOT / "tests"))
from parity import parity_report, read_golden

fixtures = {
    "homog_baseline": ModelConfig(heterogeneous_mixing=False),
    "hetero_baseline": ModelConfig(heterogeneous_mixing=True),
}
frames = {}
for fixture, config in fixtures.items():
    params = yaml.safe_load((GOLDEN / fixture / "params.yaml").read_text())["params"]
    frames[fixture] = (
        run_outputs(compile_model(config), build_outputs(config), params),
        read_golden(fixture, "derived"),
    )

## Names

Output counts in the port and the original, per fixture. The bars in each pair should be the
same height, and the assertion checks the names themselves.

In [ ]:
counts = pd.DataFrame(
    {
        fixture: {"summer4": got.shape[1], "summer2gen": gold.shape[1]}
        for fixture, (got, gold) in frames.items()
    }
).T
for fixture, (got, gold) in frames.items():
    assert sorted(got.columns) == sorted(gold.columns), fixture
counts.plot.bar(
    barmode="group",
    title="Number of named derived outputs",
    labels={"index": "fixture", "value": "outputs", "variable": "model"},
)

## Calibration targets

The outputs the calibration scores, with age mixing: port solid, original dashed.

In [ ]:
targets = [
    "tb_incidence_per100k",
    "notifications",
    "tst_posXage_15+Xreach_reachable_perc",
    "pearl_posXreach_reachable_per100k",
    "perc_prev_subclinicalXreach_reachable",
    "mixing_matrix_distance",
]
got, gold = frames["hetero_baseline"]
long = pd.concat(
    {"summer4": got.loc[1900:, targets], "summer2gen": gold.loc[1900:, targets]}, axis=1
).stack(level=[0, 1], future_stack=True).rename("value").reset_index()
long.columns = ["year", "model", "output", "value"]
fig = long.plot.line(
    x="year", y="value", color="model", line_dash="model", facet_col="output",
    facet_col_wrap=3, title="Calibration target outputs: summer4 vs original (hetero_baseline)",
)
fig.update_yaxes(matches=None, showticklabels=True)
fig

## Error over all outputs

Maximum relative error of each output, per fixture. The assertion is the parity claim.

In [ ]:
reports = {f: parity_report(g, e, atol=1e-8) for f, (g, e) in frames.items()}
for fixture, report in reports.items():
    assert report["passes"].all(), (fixture, report[~report["passes"]].head())
pd.DataFrame({f: r["max_rel"] for f, r in reports.items()}).plot.box(
    title="Max relative error per derived output",
    labels={"variable": "fixture", "value": "max relative error"},
    log_y=True,
)